# Product Price Prediction — Fine-tuned LLaMA 3.1 8B (PEFT)

In [ ]:
import os
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm

from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from datasets import load_dataset
from peft import PeftModel


from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

In [ ]:
BASE_MODEL        = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME      = "pricer"
HF_USER           = "ed-donner"
RUN_NAME          = "2024-09-13_13.04.39"
PROJECT_RUN_NAME  = f"{PROJECT_NAME}-{RUN_NAME}"
REVISION          = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"
FINETUNED_MODEL   = f"{HF_USER}/{PROJECT_RUN_NAME}"
DATASET_NAME      = f"{HF_USER}/pricer-data"

QUANT_4_BIT = False   # 8-bit → better accuracy at the cost of more VRAM
TOP_K       = 5       # number of candidate tokens for weighted prediction

# Terminal colour helpers
_C = {
    "green":  "\033[92m",
    "yellow": "\033[93m",
    "red":    "\033[91m",
    "reset":  "\033[0m",
}


In [ ]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

dataset   = load_dataset(DATASET_NAME)
train     = dataset["train"]
test      = dataset["test"]
print(f"Dataset loaded — train: {len(train):,}  test: {len(test):,}")

In [ ]:
# Quantisation config

if QUANT_4_BIT:
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_cfg = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16,
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token        = tokenizer.eos_token
tokenizer.padding_side     = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_cfg,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

fine_tuned_model = (
    PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
    if REVISION
    else PeftModel.from_pretrained(base_model, FINETUNED_MODEL)
)
fine_tuned_model.eval()
print(f"Model ready — footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def _parse_price_from_response(text: str) -> float:
    """Pull the dollar amount that follows 'Price is $' in a generated string."""
    marker = "Price is $"
    if marker not in text:
        return 0.0
    tail = text.split(marker)[1].replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", tail)
    return float(m.group()) if m else 0.0


def _token_to_float(token: str) -> float | None:
    """Return float if token is a valid positive number, else None."""
    try:
        val = float(token.strip())
        return val if val > 0 else None
    except ValueError:
        return None

In [ ]:
# Prediction

def predict_price(prompt: str, device: str = "cuda") -> float:
    """
    Weighted-average price prediction from the top-K next-token distribution.
    """
    set_seed(42)
    input_ids      = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        logits = fine_tuned_model(
            input_ids, attention_mask=attention_mask
        ).logits[:, -1, :].cpu()

    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(TOP_K)

    candidate_prices, candidate_weights = [], []
    for prob, tok_id in zip(top_probs[0], top_ids[0]):
        token = tokenizer.decode(tok_id)
        price = _token_to_float(token)
        if price is not None:
            candidate_prices.append(price)
            candidate_weights.append(prob.item())

    if not candidate_prices:
        return 0.0

    total_weight = sum(candidate_weights)
    return sum(p * w / total_weight for p, w in zip(candidate_prices, candidate_weights))

In [ ]:
# Evaluation

def _classify(abs_error: float, truth: float) -> str:
    """Colour-code a single prediction."""
    if abs_error < 40 or abs_error / truth < 0.2:
        return "green"
    if abs_error < 80 or abs_error / truth < 0.4:
        return "yellow"
    return "red"


def _log_squared_error(truth: float, guess: float) -> float:
    return (math.log(truth + 1) - math.log(max(guess, 0) + 1)) ** 2


class PricingEvaluator:
    """
    Runs a price-prediction function over a HuggingFace dataset split and
    produces a statistical report together with a four-panel diagnostic chart.
    """

    def __init__(self, predictor, dataset_split, label: str = ""):
        self.predictor = predictor
        self.split     = dataset_split
        self.label     = label or getattr(predictor, "__name__", "model")

        # Accumulated results
        self._guesses : list[float] = []
        self._truths  : list[float] = []
        self._grades  : list[str]   = []   # "green" / "yellow" / "red"
        self._sle     : list[float] = []   # squared log errors


    def _evaluate_one(self, idx: int) -> None:
        item  = self.split[idx]
        guess = self.predictor(item["text"])
        truth = item["price"]

        abs_err  = abs(guess - truth)
        grade    = _classify(abs_err, truth)
        sle      = _log_squared_error(truth, guess)

        # Short item label for console output
        snippet = item["text"].split("\n\n")[1][:25] if "\n\n" in item["text"] else item["text"][:25]

        colour = _C.get(grade, _C["reset"])
        print(
            f"{colour}[{idx+1:>4}] "
            f"pred=${guess:>9,.2f}  true=${truth:>9,.2f}  "
            f"Δ=${abs_err:>8,.2f}  {snippet}…{_C['reset']}"
        )

        self._guesses.append(guess)
        self._truths.append(truth)
        self._grades.append(grade)
        self._sle.append(sle)

    def run(self) -> "PricingEvaluator":
        for i in tqdm(range(len(self.split)), desc=f"Evaluating [{self.label}]"):
            self._evaluate_one(i)
        self._print_report()
        self._draw_charts()
        return self


In [ ]:
def _compute_stats(self) -> dict:
        y_true = np.array(self._truths)
        y_pred = np.array(self._guesses)
        n      = len(y_true)

        return {
            "n"       : n,
            "mae"     : mean_absolute_error(y_true, y_pred),
            "rmse"    : math.sqrt(mean_squared_error(y_true, y_pred)),
            "rmsle"   : math.sqrt(sum(self._sle) / n),
            "mape"    : mean_absolute_percentage_error(y_true, y_pred) * 100,
            "r2"      : r2_score(y_true, y_pred),
            "median"  : float(np.median(np.abs(y_true - y_pred))),
            "hit_good": self._grades.count("green") / n * 100,
            "hit_ok"  : (self._grades.count("green") + self._grades.count("yellow")) / n * 100,
        }

def _print_report(self) -> None:
    s = self._compute_stats()
    sep = "=" * 65
    print(f"\n{sep}")
    print(f"  Evaluation Report — {self.label}")
    print(sep)
    print(f"  Samples evaluated : {s['n']:,}")
    print(f"  MAE               : ${s['mae']:,.2f}")
    print(f"  Median abs error  : ${s['median']:,.2f}")
    print(f"  RMSE              : ${s['rmse']:,.2f}")
    print(f"  RMSLE             : {s['rmsle']:.4f}")
    print(f"  MAPE              : {s['mape']:.2f}%")
    print(f"  R²                : {s['r2']:.4f}")
    print(f"  Hit rate (good)   : {s['hit_good']:.1f}%")
    print(f"  Hit rate (ok)     : {s['hit_ok']:.1f}%")
    print(f"{sep}\n")

In [ ]:
def _draw_charts(self) -> None:
        s      = self._compute_stats()
        y_true = np.array(self._truths)
        y_pred = np.array(self._guesses)
        abs_e  = np.abs(y_true - y_pred)
        rel_e  = abs_e / np.maximum(y_true, 1e-9) * 100

        colour_vals = [
            "green" if g == "green" else ("orange" if g == "yellow" else "red")
            for g in self._grades
        ]

        fig = plt.figure(figsize=(15, 10))
        fig.suptitle(
            f"{self.label}  |  MAE=${s['mae']:,.0f}  RMSLE={s['rmsle']:.3f}  R²={s['r2']:.3f}",
            fontsize=13, fontweight="bold",
        )
        gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

        # ① Predicted vs true (scatter)
        ax0 = fig.add_subplot(gs[0, 0])
        lim = max(y_true.max(), y_pred.max()) * 1.05
        ax0.plot([0, lim], [0, lim], "--", color="royalblue", lw=1.5, alpha=0.7, label="Perfect")
        ax0.scatter(y_true, y_pred, s=8, c=colour_vals, alpha=0.55)
        ax0.set_xlim(0, lim); ax0.set_ylim(0, lim)
        ax0.set_xlabel("True price ($)"); ax0.set_ylabel("Predicted ($)")
        ax0.set_title("Predicted vs True"); ax0.legend(fontsize=8); ax0.grid(alpha=0.25)

        # ② Absolute error distribution
        ax1 = fig.add_subplot(gs[0, 1])
        ax1.hist(abs_e, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
        ax1.axvline(s["mae"],    color="crimson",  lw=1.8, linestyle="--", label=f"MAE=${s['mae']:,.0f}")
        ax1.axvline(s["median"], color="darkorange", lw=1.8, linestyle=":",  label=f"Median=${s['median']:,.0f}")
        ax1.set_xlabel("Absolute error ($)"); ax1.set_ylabel("Count")
        ax1.set_title("Absolute Error Distribution"); ax1.legend(fontsize=8); ax1.grid(alpha=0.25)

        # ③ Relative error distribution
        ax2 = fig.add_subplot(gs[1, 0])
        ax2.hist(rel_e, bins=30, color="mediumpurple", edgecolor="white", alpha=0.8)
        ax2.axvline(s["mape"], color="crimson", lw=1.8, linestyle="--", label=f"MAPE={s['mape']:.1f}%")
        ax2.set_xlabel("Relative error (%)"); ax2.set_ylabel("Count")
        ax2.set_title("Relative Error Distribution"); ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

        # ④ Mean absolute error by price bucket
        ax3 = fig.add_subplot(gs[1, 1])
        edges  = np.percentile(y_true, np.linspace(0, 100, 6))
        labels = [f"${edges[i]:.0f}–\n${edges[i+1]:.0f}" for i in range(len(edges) - 1)]
        bucket = np.digitize(y_true, edges[1:-1])   # 0-indexed bucket per sample
        bucket_mae = [abs_e[bucket == b].mean() if (bucket == b).any() else 0.0
                      for b in range(len(labels))]
        bars = ax3.bar(labels, bucket_mae, color="seagreen", alpha=0.8, edgecolor="white")
        ax3.bar_label(bars, fmt="$%.0f", padding=3, fontsize=8)
        ax3.set_ylabel("Mean absolute error ($)")
        ax3.set_title("MAE by Price Quintile"); ax3.grid(alpha=0.25, axis="y")

        plt.savefig("pricer_eval_report.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Chart saved → pricer_eval_report.png")

In [ ]:
# Entry point

evaluator = PricingEvaluator(
    predictor=predict_price,
    dataset_split=test,
    label=f"LLaMA-3.1-8B Fine-tuned | {'4-bit' if QUANT_4_BIT else '8-bit'} quant | top-{TOP_K} weighted",
)
evaluator.run()